# Module 10: Wednesday Session
## CS 5045: Computation for the Data Sciences

**Topic:** Multidimensional Data & GroupBy

This notebook follows the session plan. Each section corresponds to a demo segment.
Pause at each **Pause and Ask** prompt before running the next cell.

---
## Setup

**Operation: Represent**

We always start by creating our data and checking its shape.
This is a small HR dataset defined inline, no CSV file required.
Notice: 16 rows, 4 departments, 2 locations, 2 job levels.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# REPRESENT: build the HR dataset inline
data = {
    'employee_id': list(range(1, 17)),
    'department':  ['Engineering', 'Engineering', 'Engineering', 'Engineering',
                    'Marketing',   'Marketing',   'Marketing',   'Marketing',
                    'Sales',       'Sales',       'Sales',       'Sales',
                    'HR',          'HR',          'HR',          'HR'],
    'location':    ['East', 'East', 'West', 'West',
                    'East', 'East', 'West', 'West',
                    'East', 'East', 'West', 'West',
                    'East', 'East', 'West', 'West'],
    'job_level':   ['Senior', 'Junior', 'Senior', 'Junior',
                    'Senior', 'Junior', 'Senior', 'Junior',
                    'Senior', 'Junior', 'Senior', 'Junior',
                    'Senior', 'Junior', 'Senior', 'Junior'],
    'salary':      [95000, 72000, 98000, 74000,
                    82000, 65000, 85000, 67000,
                    78000, 62000, 80000, 64000,
                    72000, 58000, 74000, 60000],
    'tenure_years': [5, 2, 7, 3, 4, 1, 6, 2, 3, 1, 5, 2, 4, 1, 6, 3]
}

df = pd.DataFrame(data)

print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
print()
print('Unique values per categorical column:')
for col in ['department', 'location', 'job_level']:
    print(f'  {col}: {sorted(df[col].unique())}')
print()
df

---
## Part 1: Single-Dimension GroupBy

**Operation: Aggregate**

Question: *What is the mean salary for each department?*

Before running the cell below, notice that `df.groupby('department')` alone returns a
GroupBy *object*, not a result. The Apply step, `.mean()`, `.sum()`, etc., is what
triggers the computation.

The pattern: **Split** (groupby) -> **Apply** (.mean) -> **Combine** (pandas assembles the result).

In [ ]:
# Show the GroupBy object BEFORE applying a function
grp_obj = df.groupby('department')
print(type(grp_obj))   # DataFrameGroupBy -- not a result yet
print(grp_obj)         # shows what groups exist
print()

In [ ]:
# AGGREGATE: split by department, compute mean salary within each group
by_dept = df.groupby('department')['salary'].mean()
print('Mean salary by department:')
print(by_dept)
print()
print(f'Result type: {type(by_dept).__name__}')   # Series
print(f'Number of groups: {len(by_dept)}')

---
## Part 2: Multidimensional GroupBy

**Operation: Aggregate (multiple dimensions)**

Question: *What is the mean salary for each combination of department AND location?*

The only change from Part 1: pass a **list** of column names instead of a single string.
Pandas creates groups for every unique combination of values.

> **Pause and Ask:** With 4 departments and 2 locations, how many groups do you expect?
> Type your answer in chat.
> *(Answer: 4 x 2 = 8)*

In [ ]:
# AGGREGATE: two-dimension groupby
by_dept_loc = df.groupby(['department', 'location'])['salary'].mean()
print('Mean salary by department AND location:')
print(by_dept_loc)
print()
print(f'Number of groups: {len(by_dept_loc)}')
print(f'Index type: {type(by_dept_loc.index).__name__}')   # MultiIndex

---
## Part 3: Multiple Statistics with .agg()

**Operation: Aggregate (multiple outputs)**

Question: *For each department, what is the mean salary, headcount, and longest tenure?*

`.agg()` lets you compute multiple statistics in a single GroupBy call.
Each keyword argument becomes a column in the result.
The format for each argument is a tuple: `(source_column, function_name)`.

> **Pause and Ask:** What fourth statistic would you add to this summary for your own
> project dataset? Type a suggestion in chat.

In [ ]:
# AGGREGATE: multiple statistics in one call
dept_summary = df.groupby('department').agg(
    mean_salary  = ('salary',       'mean'),
    headcount    = ('employee_id',  'count'),
    max_tenure   = ('tenure_years', 'max')
)
print('Department summary: mean salary, headcount, longest tenure:')
print(dept_summary)
print()
print('Sorted by mean salary (highest first):')
print(dept_summary.sort_values('mean_salary', ascending=False))

---
## Part 4: Pivot Tables with .unstack()

**Operations: Aggregate then Transform**

After groupby with two columns, the result has a MultiIndex, two levels of row labels.
This is correct but hard to scan as a long list.
`.unstack()` rotates one index level into column headers, creating a cross-tabular pivot table.

> **Pause and Ask:** Before I run `.unstack()`, we have 8 rows. After I run it, how many
> rows will we have? Type in chat.
> *(Answer: 4, departments become rows, job levels become columns)*

In [ ]:
# AGGREGATE: group by department and job level
grouped = df.groupby(['department', 'job_level'])['salary'].mean()
print('Before unstack (MultiIndex Series):')
print(grouped)
print(f'Shape: {grouped.shape}')

In [ ]:
# TRANSFORM: rotate the innermost index level (job_level) into column headers
pivot = grouped.unstack(fill_value=0)
print('After unstack (pivot table, departments as rows, job levels as columns):')
print(pivot)
print(f'Shape: {pivot.shape}')
print()
print('Notice: .stack() would reverse this operation.')

---
## Part 5: Filter + Aggregate

**Operations: Filter then Aggregate**

Most real questions are about a specific subset of data.
The pattern: **Filter first** (narrow the rows) -> **Aggregate** (summarize the subset).

Question: *Among Senior employees only, what is the mean salary by department?*

In [ ]:
# FILTER: keep only Senior employees
seniors = df[df['job_level'] == 'Senior']
print(f'After filter: {len(seniors)} rows (down from {len(df)})')
print(seniors[['employee_id', 'department', 'location', 'job_level', 'salary']])

In [ ]:
# AGGREGATE: mean salary by department within the Senior subset
senior_salary = seniors.groupby('department')['salary'].mean()
print('Mean salary for Senior employees, by department:')
print(senior_salary)
print()
print('Compare to mean salary across ALL employees:')
all_salary = df.groupby('department')['salary'].mean()
comparison = pd.DataFrame({'senior_mean': senior_salary, 'all_mean': all_salary})
comparison['senior_premium'] = (comparison['senior_mean'] - comparison['all_mean']).astype(int)
print(comparison)

---
## CLI Threading: The Notebook Is a Tool

Run the cell below to save `data.csv` to your working directory.
Then open a terminal and paste the command printed at the end.

The output is the same as what Part 1 produced: same groupby, same result, different environment.

> **The notebook is a tool. The computation is Python.**

In [ ]:
# Save the dataset so the terminal demo can read it
df.to_csv('data.csv', index=False)
print('data.csv saved to working directory.')
print()
print('Open a terminal. Paste this command and run it:')
print()
print('python -c "import pandas as pd; df = pd.read_csv(\'data.csv\'); print(df.groupby(\'department\')[\'salary\'].mean().to_string())"')
print()
print('Expected output:')
print(df.groupby('department')['salary'].mean().to_string())

---
## Connecting to the Assignment (hw10.ipynb)

`hw10.ipynb` is mostly a fill-in-the-blanks notebook. Most blanks are marked `__________`;
fill in the Python expressions and the notebook should run without errors when complete.
Section 5 is the exception: no blanks, just a question. The dataset is **retail sales data**
(region, product_category, quarter, units_sold, revenue) instead of HR data. Same operations,
different context.

**Section by section:**

**Section 1: Basic GroupBy**
- Fill in the groupby column name and the aggregation function (same as Part 1 above)
- Use `.idxmax()` to find the region with the highest total revenue
- Index a specific region's unit count by name

**Section 2: Multiple Statistics with .agg()**
- Fill in two function name strings ('mean' and 'sum') in the `.agg()` call (same as Part 3)
- Call `.reset_index()` to flatten the MultiIndex result into a plain DataFrame

**Section 3: Grouped Bar Chart**
- Fill in the `.plot()` call parameters: kind='bar', title, and ylabel
- Fill in two groupby column names and chain `.unstack()` to create a multi-category chart

**Section 4: Multi-Level GroupBy**
- Fill in two groupby column names to group by both region and quarter (same as Part 4)
- Chain `.unstack(fill_value=0)` to produce the pivot table
- Index the Q2 column and use `.idxmax()` to find the top Q2 region

**Section 5: Code From Scratch**
- No blanks. You're given a question only: compose Filter + GroupBy + Aggregate yourself,
  the same way Part 5 above filtered to Senior employees before aggregating.
- One-sentence reflection question follows the code.

**Section 6: Interpret**
- Two Markdown cells for written answers, plain English, no Python syntax

**The only new wrinkle:** Section 3 chains `.unstack()` before `.plot()` to get a grouped
bar chart. That is the same `.unstack()` from Part 4 above. Write the two steps on separate
lines and check the shape before plotting.